# User-Based CF (Temporal Split - 99% Coverage)

## Modified Strategy

This notebook uses **temporal split data** BUT filters the test set to achieve **99% coverage**:
- **Training**: 80% oldest ratings (same as original temporal split)
- **Testing**: From 20% newest ratings, **filter to only user-movie pairs where BOTH exist in training**

**Purpose**: Compare algorithm performance on temporal data WITHOUT cold-start complications.

**Key Difference from Original Temporal Split**:
- Original: 8.97% coverage (89% cold-start users)
- This version: ~99% coverage (filters out cold-start pairs)

**Note**: This is NOT realistic for production, but allows fair algorithm comparison on temporal data.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm
import time
import gc
import psutil
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Load Temporal Split Data

In [ ]:
# Load temporal split data
train_path = '../../datasets/output/split_and_train_datasets/temporal_split/train_ratings.csv'
test_path = '../../datasets/output/split_and_train_datasets/temporal_split/test_ratings.csv'

print("=" * 60)
print("LOADING TEMPORAL SPLIT DATA (FILTERED FOR 99% COVERAGE)")
print("=" * 60)
print("\nStrategy: Temporal split + coverage filter")
print("  - Train: 80% oldest ratings")
print("  - Test: 20% newest ratings (FILTERED to known user-movie pairs)")
print("  - Purpose: Fair algorithm comparison on temporal data")
print()

print("Loading training data...")
train = pd.read_csv(train_path)
print(f"Train shape: {train.shape}")

print("\nLoading test data...")
test = pd.read_csv(test_path)
print(f"Test shape: {test.shape}")

print("\nData loaded successfully!")
print("=" * 60)

In [ ]:
# Dataset statistics
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

print("\nTraining Set:")
print(f"  Unique users: {train['userId'].nunique():,}")
print(f"  Unique movies: {train['movieId'].nunique():,}")
print(f"  Total ratings: {len(train):,}")
print(f"  Mean rating: {train['rating'].mean():.2f}")

print("\nTest Set (BEFORE filtering):")
print(f"  Unique users: {test['userId'].nunique():,}")
print(f"  Unique movies: {test['movieId'].nunique():,}")
print(f"  Total ratings: {len(test):,}")
print(f"  Mean rating: {test['rating'].mean():.2f}")

## 3. Create User-Item Matrix with Mean-Centering

In [ ]:
# Create ID mappings from TRAINING data only
print("Creating ID mappings from TRAINING data...")

unique_users = train['userId'].unique()
unique_movies = train['movieId'].unique()

user_id_map = {id: idx for idx, id in enumerate(unique_users)}
movie_id_map = {id: idx for idx, id in enumerate(unique_movies)}

idx_to_user = {idx: id for id, idx in user_id_map.items()}
idx_to_movie = {idx: id for id, idx in movie_id_map.items()}

print(f"Training users: {len(user_id_map):,}")
print(f"Training movies: {len(movie_id_map):,}")

In [ ]:
# Calculate per-user mean ratings (for mean-centering)
print("\nCalculating per-user mean ratings...")

user_mean_ratings = train.groupby('userId')['rating'].mean().to_dict()
global_mean_rating = train['rating'].mean()

print(f"Global mean rating: {global_mean_rating:.3f}")
print(f"User mean rating range: [{min(user_mean_ratings.values()):.2f}, {max(user_mean_ratings.values()):.2f}]")

# Apply mean-centering to training data
print("\nApplying mean-centering to training data...")
train['rating_centered'] = train.apply(
    lambda x: x['rating'] - user_mean_ratings.get(x['userId'], global_mean_rating),
    axis=1
)

print(f"Centered rating range: [{train['rating_centered'].min():.2f}, {train['rating_centered'].max():.2f}]")
print(f"Centered rating mean: {train['rating_centered'].mean():.4f} (should be ~0)")

In [ ]:
# Map train data to indices
print("\nMapping training data to matrix indices...")
train['user_idx'] = train['userId'].map(user_id_map)
train['movie_idx'] = train['movieId'].map(movie_id_map)

# Map test data to indices
print("Mapping test data to matrix indices...")
test['user_idx'] = test['userId'].map(user_id_map)
test['movie_idx'] = test['movieId'].map(movie_id_map)

# CRITICAL: Filter test set to only user-movie pairs that exist in training
print("\n" + "=" * 60)
print("FILTERING TEST SET FOR 99% COVERAGE")
print("=" * 60)

test_before_filter = len(test)
testable = test.dropna(subset=['user_idx', 'movie_idx']).copy()
test_after_filter = len(testable)

print(f"\nTest ratings before filter: {test_before_filter:,}")
print(f"Test ratings after filter: {test_after_filter:,}")
print(f"Removed (cold-start): {test_before_filter - test_after_filter:,}")
print(f"Coverage achieved: {test_after_filter/test_before_filter*100:.2f}%")

# Store for later use
true_coverage = test_after_filter / test_before_filter * 100
total_testable = test_after_filter

print(f"\n✓ Test set filtered to {total_testable:,} testable ratings")

In [ ]:
# Create sparse user-item matrices
print("\nCreating sparse user-item matrices...")

start_time = time.time()

# CSR format for row access (users)
user_item_matrix_csr = csr_matrix(
    (train['rating_centered'].values,
     (train['user_idx'].values, train['movie_idx'].values)),
    shape=(len(user_id_map), len(movie_id_map))
)

# CSC format for column access (movies) - MUCH FASTER for getting movie columns
print("Converting to CSC format for fast movie access...")
user_item_matrix_csc = user_item_matrix_csr.tocsc()

elapsed = time.time() - start_time

print(f"\nMatrices created in {elapsed:.2f} seconds")
print(f"Shape: {user_item_matrix_csr.shape}")
print(f"Memory (CSR): {user_item_matrix_csr.data.nbytes / (1024**2):.2f} MB")
print(f"Memory (CSC): {user_item_matrix_csc.data.nbytes / (1024**2):.2f} MB")

## 4. Pre-compute User Norms

In [ ]:
print("=" * 60)
print("PRE-COMPUTING USER NORMS")
print("=" * 60)

n_users = user_item_matrix_csr.shape[0]
batch_size = 10000
user_norms = np.zeros(n_users)

print(f"Processing {n_users:,} users in batches of {batch_size:,}...")

for batch_start in tqdm(range(0, n_users, batch_size), desc="Computing norms"):
    batch_end = min(batch_start + batch_size, n_users)
    batch_matrix = user_item_matrix_csr[batch_start:batch_end]
    batch_norms = np.sqrt(batch_matrix.multiply(batch_matrix).sum(axis=1).A1)
    user_norms[batch_start:batch_end] = batch_norms
    del batch_matrix, batch_norms
    if (batch_start // batch_size) % 5 == 0:
        gc.collect()

gc.collect()

print(f"\n✓ Pre-computed {len(user_norms):,} user norms")
print(f"  Memory: {user_norms.nbytes / (1024**2):.2f} MB")

## 5. Prediction Function

In [ ]:
def predict_rating(user_idx, movie_idx, k=50, max_candidates=200):
    """
    User-Based CF prediction with on-demand similarity computation.
    
    Parameters:
    - user_idx: Index of user
    - movie_idx: Index of movie  
    - k: Number of similar users to consider (default: 50)
    - max_candidates: Max users to consider per prediction (default: 200)
    
    Returns:
    - predicted_rating: Float [0.5, 5.0]
    """
    user_idx = int(user_idx)
    movie_idx = int(movie_idx)
    
    # Get users who rated this movie (using CSC for efficient column access)
    movie_col = user_item_matrix_csc[:, movie_idx]
    users_who_rated = movie_col.nonzero()[0]
    
    if len(users_who_rated) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Remove target user from candidates
    users_who_rated = users_who_rated[users_who_rated != user_idx]
    
    if len(users_who_rated) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Limit candidates for efficiency
    if len(users_who_rated) > max_candidates:
        users_who_rated = np.random.choice(users_who_rated, max_candidates, replace=False)
    
    # Get target user's rating vector
    target_user_row = user_item_matrix_csr[user_idx]
    target_norm = user_norms[user_idx]
    
    if target_norm == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Compute similarities on-demand
    similarities = []
    ratings_centered = []
    
    for other_user_idx in users_who_rated:
        other_norm = user_norms[other_user_idx]
        if other_norm == 0:
            continue
        
        dot_product = target_user_row.dot(user_item_matrix_csr[other_user_idx].T).toarray()[0, 0]
        similarity = dot_product / (target_norm * other_norm)
        rating_centered = movie_col[other_user_idx, 0]
        
        similarities.append(similarity)
        ratings_centered.append(rating_centered)
    
    if len(similarities) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    similarities = np.array(similarities)
    ratings_centered = np.array(ratings_centered)
    
    # Select top-k most similar users
    if len(similarities) > k:
        top_k_indices = np.argpartition(similarities, -k)[-k:]
        top_k_indices = top_k_indices[np.argsort(similarities[top_k_indices])[::-1]]
        top_k_sims = similarities[top_k_indices]
        top_k_ratings = ratings_centered[top_k_indices]
    else:
        top_k_sims = similarities
        top_k_ratings = ratings_centered
    
    # Filter out near-zero similarities
    valid = np.abs(top_k_sims) > 1e-6
    top_k_sims = top_k_sims[valid]
    top_k_ratings = top_k_ratings[valid]
    
    if len(top_k_sims) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Compute weighted average
    weighted_sum = np.sum(top_k_sims * top_k_ratings)
    sum_of_weights = np.sum(np.abs(top_k_sims))
    
    if sum_of_weights == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    predicted_centered = weighted_sum / sum_of_weights
    
    # Denormalize: add back user's mean rating
    user_id = idx_to_user[user_idx]
    user_mean = user_mean_ratings.get(user_id, global_mean_rating)
    predicted = predicted_centered + user_mean
    
    return np.clip(predicted, 0.5, 5.0)

print("✓ Prediction function defined!")

## 6. Generate Predictions

In [ ]:
# Configuration
SAMPLE_SIZE = 100000  # 100K for fair comparison
BATCH_SIZE = 1000
GC_FREQUENCY = 5

# Sample from testable ratings
if SAMPLE_SIZE and SAMPLE_SIZE < len(testable):
    print(f"Sampling {SAMPLE_SIZE:,} from {len(testable):,} testable ratings")
    test_sample = testable.sample(SAMPLE_SIZE, random_state=42)
else:
    print(f"Using all {len(testable):,} testable ratings")
    test_sample = testable.copy()

print(f"\nGenerating predictions for {len(test_sample):,} ratings...")
print(f"Estimated time: 30-60 minutes")

mem_before = psutil.Process().memory_info().rss / (1024**3)
print(f"Memory before: {mem_before:.2f} GB\n")

start_time = time.time()
predictions = []
n_batches = (len(test_sample) + BATCH_SIZE - 1) // BATCH_SIZE
test_rows = test_sample[['user_idx', 'movie_idx']].values

for batch_idx in tqdm(range(n_batches), desc="Processing batches"):
    batch_start = batch_idx * BATCH_SIZE
    batch_end = min((batch_idx + 1) * BATCH_SIZE, len(test_rows))
    
    batch_predictions = []
    for i in range(batch_start, batch_end):
        user_idx, movie_idx = test_rows[i]
        pred = predict_rating(user_idx, movie_idx, k=50, max_candidates=200)
        batch_predictions.append(pred)
    
    predictions.extend(batch_predictions)
    
    if batch_idx % GC_FREQUENCY == 0:
        gc.collect()

gc.collect()
elapsed = time.time() - start_time
mem_after = psutil.Process().memory_info().rss / (1024**3)

print(f"\n✓ Predictions completed in {elapsed/60:.2f} minutes")
print(f"  Average: {elapsed/len(test_sample)*1000:.2f} ms per rating")
print(f"  Memory: {mem_after:.2f} GB")

## 7. Evaluation

In [ ]:
# Calculate metrics
test_sample = test_sample.copy()
test_sample['predicted_rating'] = predictions

actual = test_sample['rating'].values
predicted = test_sample['predicted_rating'].values

rmse = np.sqrt(mean_squared_error(actual, predicted))
mae = mean_absolute_error(actual, predicted)

print("="*60)
print("USER-BASED CF RESULTS (TEMPORAL SPLIT - 99% COVERAGE)")
print("="*60)
print(f"\nAlgorithm: User-Based CF")
print(f"Split: Temporal (filtered for known user-movie pairs)")
print(f"Similarity: Cosine on centered ratings")
print(f"k-neighbors: 50")
print(f"Max candidates: 200")

print(f"\nPerformance:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")

print(f"\nTiming:")
print(f"  Training: {elapsed/60:.2f} minutes")
print(f"  Per rating: {elapsed/len(test_sample)*1000:.2f} ms")

print(f"\nCoverage:")
print(f"  Coverage: {true_coverage:.2f}%")
print(f"  Test samples: {len(test_sample):,}")
print(f"  Total testable: {total_testable:,}")
print("="*60)

## 8. Save Results

In [ ]:
# Save results
results = {
    'algorithm': 'User-Based CF (Temporal-99)',
    'split_strategy': 'temporal (filtered)',
    'similarity_metric': 'Cosine (centered)',
    'normalization': 'Mean-centering per user',
    'k_neighbors': 50,
    'max_candidates': 200,
    'rmse': rmse,
    'mae': mae,
    'coverage': true_coverage,
    'training_time_minutes': elapsed/60,
    'prediction_time_ms': elapsed/len(test_sample)*1000,
    'test_samples': len(test_sample),
    'total_testable': total_testable
}

results_df = pd.DataFrame([results])
output_path = '../../datasets/output/model_implementations/user_based_cf_temporal_99_coverage.csv'
results_df.to_csv(output_path, index=False)

print(f"✓ Results saved to: {output_path}")
print("\nResults Summary:")
print(results_df.T)

## Summary

This notebook evaluates User-Based CF on **temporal split data with 99% coverage**.

**Key Points**:
- Uses temporal train/test split (time-ordered)
- Filters test set to only known user-movie pairs
- Achieves ~99% coverage (vs 9% in original temporal split)
- Allows fair algorithm comparison on temporal data

**Comparison Purpose**:
- Compare with 80-20 random split (different split, same coverage)
- Shows if temporal ordering affects algorithm performance
- Isolates algorithm quality from cold-start effects